<a href="https://colab.research.google.com/github/KevalSolanki77/PCA_from_scartch_Using_Math/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

In [3]:

(X, y) = make_blobs(
    n_samples=500,
    n_features=3,
    centers=3,
    cluster_std=2,
    random_state=42
)

df = pd.DataFrame(X, columns=['X1', 'X2', 'X3'])
df['target'] = y

df.head()

,X1,X2,X3,target
0,-10.641529,4.199909,6.025774,2
1,-3.658616,8.171290,5.319521,0
2,0.526492,-2.011931,-10.420596,1
3,-9.501240,7.744464,1.693174,2
4,-6.093288,8.120739,7.049725,0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:3], df.iloc[:,3], test_size=0.2, random_state=0)
X_train.shape, X_test.shape

((400, 3), (100, 3))

In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier()

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("Accuracy Score : ", accuracy_score(y_pred, y_test))

Accuracy Score :  0.98


In [9]:
import plotly.express as px

fig = px.scatter_3d(data_frame=df, x=df['X1'], y = df['X2'], z = df['X3'], color = df['target'].astype("str"))

fig.update_traces(marker = dict(size = 12,
                                line = dict(width = 2,
                                            color = 'DarkSlateGrey')),
                  selector=dict(mode = 'markers'))

fig.show()

This is how the data looks in the 3D to convert this into 2D just imagine you are clicking the picture of this data. The picture will be in the 2D, so we have to take the picture from an angle where the data's spread is similar to the 3D representation.

---

To convert this 3D data into 2D we have to find the Covariance Matrix, Eigen Vectors and Eigen Values

---
# Step 1: Mean Centering

The first step of the PCA is to do Mean Centering the data which can be done by StanderScaler


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 2: Build Covariance Matrix

In [11]:
cov_matrix = np.cov([df['X1'], df['X2'], df['X3']])
print("Covariance Matrix : \n", cov_matrix)

Covariance Matrix : 
 [[ 24.05232732 -24.30544447 -14.95333027]
 [-24.30544447  56.69007513  36.10296322]
 [-14.95333027  36.10296322  29.30837846]]


# Step 3: Find Eigen Vectors and Eigen Values

In [14]:
eigen_values, eigen_vectors = np.linalg.eig(cov_matrix)
print("Eigen Values : ", eigen_values)
print("Eigen Vectors : \n", eigen_vectors)

Eigen Values :  [93.33654508 12.56591647  4.14831936]
Eigen Vectors : 
 [[ 0.38050291 -0.91133744 -0.1571038 ]
 [-0.76470071 -0.21453143 -0.60762578]
 [-0.52004842 -0.35134076  0.77853022]]


# Step 4 : Selecting the 2 Eigen Vectors with highest Eigen Values

Those vectors will be my Principal Components

In [16]:
pc = eigen_vectors[:2]
pc

array([[ 0.38050291, -0.91133744, -0.1571038 ],
       [-0.76470071, -0.21453143, -0.60762578]])

# Step 5 : Dot Product of the data and Transpose of Principal Components

In [21]:
transformed_data = np.dot(df.iloc[:, :3], pc.T)

In [22]:
new_df = pd.DataFrame(transformed_data, columns = ['PC1', 'PC2'])
new_df['target'] = df['target'].values
new_df.head()

,PC1,PC2,target
0,-8.823339,3.575156,2
1,-9.674633,-2.187530,0
2,3.670995,6.360837,1
3,-10.939074,4.575358,2
4,-10.826786,-1.366207,0


This is the 2D representation of the 3D data we can also convert this into 1D by taking only Eigen Vector with the highest Eigen Value

---

In [24]:
new_X_train, new_X_test, new_y_train, new_y_test = train_test_split(new_df.iloc[:,:2], new_df.iloc[:,2], test_size=0.2, random_state=0)

In [26]:
fig = px.scatter(x=new_X_train['PC1'],
           y=new_X_train['PC2'],
           color = new_y_train.astype(str),
           color_discrete_sequence = px.colors.qualitative.G10)
fig.show()

In [29]:
knn.fit(new_X_train, new_y_train)

y_pred = knn.predict(new_X_test)

print("Accuracy Score : ", accuracy_score(y_pred, y_test))

Accuracy Score :  0.96


See Decreases because of the too much reduction in Dimentionality but the point is you can make your own PCA using Math.

Now imagine you are working with the MNIST dataset where input columns are 784 (28x28 pixel images) where computation will increase if the take the dataset as it is there it will be helpful